# Data Preprocessing

## Imports

In [73]:
import pandas as pd
cleaned_data_path = "../data/cleaned_data/"
model_data_path = "../data/modeling_data/"

def save_model_data(df, file_name):
    df.to_csv(model_data_path + file_name, index=False)

## online_retail_data

In [74]:
online_retail_data = pd.read_csv(cleaned_data_path + "online_retail_data_clean.csv")
online_retail_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Feature Engineering:
- Totalprice = Quantity * UnitPrice
- InvoiceDate = Date, Time, Month, DayofWeek

In [75]:
online_retail_data['TotalPrice'] = (
    online_retail_data['Quantity'] * online_retail_data['UnitPrice']
)
online_retail_data['InvoiceDate'] = pd.to_datetime(
    online_retail_data['InvoiceDate'], errors='coerce'
)
online_retail_data['InvoiceHour'] = online_retail_data['InvoiceDate'].dt.hour
online_retail_data['InvoiceDay'] = online_retail_data['InvoiceDate'].dt.day
online_retail_data['InvoiceMonth'] = online_retail_data['InvoiceDate'].dt.month
online_retail_data['InvoiceDow'] = online_retail_data['InvoiceDate'].dt.dayofweek
online_retail_data['IsCancelled'] = (
    online_retail_data['InvoiceNo'].astype(str).str.startswith('C')
)

online_retail_data.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,InvoiceHour,InvoiceDay,InvoiceMonth,InvoiceDow,IsCancelled
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,8,1,12,2,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,8,1,12,2,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2,False


### New Features

We define a new target variable called ```HIGH_VALUE_CUSTOMER``` based on customer purchasing behavior.

This label is constructed using a rule that combines lifetime spending and purchase frequency.

The rule is used only to assign the target variable, not as an input to any model.

The machine learning models will never receive the variables or thresholds used in this rule to avoid data leakage.

Instead, models must learn to predict ```HIGH_VALUE_CUSTOMER``` using independent behavioral features (recency, frequency, average basket size, etc.).

This setup ensures that model evaluation reflects true predictive ability, not trivial reconstruction of the rule.

In [76]:
g = online_retail_data.groupby("CustomerID")

# LABEL COMPONENTS (used ONLY for y)
lifetime_spend = g["TotalPrice"].sum()
spend_threshold = lifetime_spend.quantile(0.50)
big_spender = lifetime_spend >= spend_threshold

avg_interpurchase = g["InvoiceDate"].apply(
    lambda x: x.sort_values().diff().dt.days.mean()
)
engaged = avg_interpurchase < 4   # frequent buyers

# FINAL LABEL
label = (big_spender & engaged).astype(int)

# FEATURES (safe, no leakage)
max_date = online_retail_data["InvoiceDate"].max()

customer_features = pd.DataFrame({
    "Recency": (max_date - g["InvoiceDate"].max()).dt.days,
    "Frequency": g["InvoiceNo"].nunique(),
    "UniqueItems": g["Description"].nunique(),
    "AvgBasketQty": g["Quantity"].sum() / g["InvoiceNo"].nunique(),
    "AvgUnitPrice": g["UnitPrice"].mean(),
    "CancelRatio": g["InvoiceNo"].apply(lambda x: x.astype(str).str.startswith("C").mean()),
})

# ADD LABEL (aligned automatically by CustomerID index)
customer_features["HighValueCustomer"] = label

customer_features = customer_features.reset_index()


customer_features["HighValueCustomer"].value_counts(normalize=True)



HighValueCustomer
0    0.594465
1    0.405535
Name: proportion, dtype: float64

In [77]:
customer_features.head()

,CustomerID,Recency,Frequency,UniqueItems,AvgBasketQty,AvgUnitPrice,CancelRatio,HighValueCustomer
0,12346.0,325,2,1,0.000000,1.040000,0.5,0
1,12347.0,1,7,103,351.142857,2.644011,0.0,1
2,12348.0,74,4,22,585.250000,5.764839,0.0,0
3,12349.0,18,1,73,631.000000,8.289041,0.0,1
4,12350.0,309,1,17,197.000000,3.841176,0.0,0


In [78]:
save_model_data(customer_features, "customer_features_model.csv")

## Wine Dataset

- We combine the Red Wine and White Wine Quality datasets from UCI into a single unified dataset.
- A new binary target variable `is_red` is created:
  - `1` = red wine
  - `0` = white wine
- All original chemical features (acidity, chlorides, sulfur dioxide, alcohol, etc.) are kept as input features.
- The target variable `is_red` is used for a binary classification task:
  - Predict whether a wine sample is red or white based solely on its chemical properties.
- No data splitting or preprocessing occurs during merging; the dataset is simply concatenated and exported.
- The merged dataset becomes a clean, ready-to-use input for evaluating and comparing multiple classifiers.
- This classification task is leak-free and typically achieves high accuracy, making it ideal for demonstrating model performance differences.


In [79]:
red = pd.read_csv(cleaned_data_path + "wine_red_data_clean.csv")
white = pd.read_csv(cleaned_data_path + "wine_white_data_clean.csv")

# Add binary label: 1 = red, 0 = white
red["is_red"] = 1
white["is_red"] = 0

# Merge into one unified dataset
wine_combined = pd.concat([red, white], ignore_index=True)

In [80]:
save_model_data(wine_combined, "wine_combined_data_model.csv")

## Cup98

- The CUP98 dataset is already structured as a supervised learning problem with a built-in binary target variable, `TARGET_B`, which indicates whether a customer made a donation.
- Because the target is clearly defined and widely used in prior work—including the paper we reference—we do not need to engineer a new label.
- The research study explicitly uses CUP98 “as is,” evaluating classifiers directly on `TARGET_B`, so we follow the same methodology for consistency.
- CUP98 contains a rich set of demographic and historical donation features, which can be used directly without additional feature engineering.
- Apart from removing ID-like or leakage columns, the dataset requires no modification before modeling.
- Using the dataset as provided ensures a fair, standard comparison across models and aligns with established practices in the literature.


### Addressing Possible Leakage

- The CUP98 dataset includes several variables that were collected **after** the mailing campaign or were never intended for prediction; these are considered **leakage features** because they contain information that would not be available at prediction time.
- Even though these leakage variables appear in the dataset, **using them in a predictive model is not valid**, because it provides the model with unfair knowledge about the outcome.
- Examples include:
  - Unique identifiers (e.g., CONTROLN), which carry no predictive value and can cause the model to memorize rows.
  - Mailing dates or campaign metadata (e.g., MAILDATE) that were assigned after segmentation and may correlate with donation outcomes.
  - Administrative flags (e.g., HPHONE_D) that may reflect internal processing decisions that occurred after the donation behavior.
- Using leakage features leads to **inflated accuracy** during evaluation but produces a model that fails in real deployment because it relies on information it would never have at prediction time.
- For this reason, even though the dataset includes these variables, we **intentionally remove them** to ensure a fair, realistic supervised learning setup that matches the methodology used in prior research.


In [81]:
cup98 = pd.read_csv(cleaned_data_path + "cup98_data_clean.csv")

drop_cols = ["CONTROLN", "MAILDATE", "HPHONE_D"]
drop_cols = [c for c in drop_cols if c in cup98.columns]

cup98 = cup98.drop(columns=drop_cols)


### One-Hot Encoding Categories

In [82]:
if "TARGET_B" not in cup98.columns:
    raise ValueError("TARGET_B not found in CUP98 dataset.")

y = cup98["TARGET_B"]
X = cup98.drop(columns=["TARGET_B"])



In [83]:
cup98.head()


,ODATEDW,OSOURCE,TCODE,STATE,ZIP,MAILCODE,PVASTATE,DOB,NOEXCH,RECINHSE,...,TARGET_B,TARGET_D,RFA_2R,RFA_2F,RFA_2A,MDMAUD_R,MDMAUD_F,MDMAUD_A,CLUSTER2,GEOCODE2
0,8901,GRI,0,IL,61081.0,MISSING,MISSING,3712,0.0,MISSING,...,0,0.0,L,4,E,MISSING,1.568027,MISSING,39.0,C
1,9401,BOA,1,CA,91326.0,MISSING,MISSING,5202,0.0,MISSING,...,0,0.0,L,2,G,MISSING,1.568027,MISSING,1.0,A
2,9001,AMH,1,NC,27017.0,MISSING,MISSING,0,0.0,MISSING,...,0,0.0,L,4,E,MISSING,1.568027,MISSING,60.0,C
3,8701,BRY,0,CA,95953.0,MISSING,MISSING,2801,0.0,MISSING,...,0,0.0,L,4,E,MISSING,1.568027,MISSING,41.0,C
4,8601,MISSING,0,FL,33176.0,MISSING,MISSING,2001,0.0,MISSING,...,0,0.0,L,2,F,MISSING,1.568027,MISSING,26.0,A
